# Agent implementation moved into this notebook

This notebook now contains the RCA agent implementation that used to live in `multiAgentSystem/agent.py`.

- Run cells below to import and instantiate `RCAAgent`.
- This makes it easier to iterate in Databricks notebooks and keeps the runnable code in one place.


# Multi-Agent System for Root Cause Analysis

This notebook contains the main agent wrapper and MLflow integration for the multi-agent system.


In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
# Import the RCAAgent class and AGENT instance from the last cell
# The RCAAgent class is defined in the cell below
# For now, we'll just verify the imports work
import sys
sys.path.append('/Users/amruth.ashok/Desktop/Amruth/mutliAgent/Spark-RCA-assistant')

# Import dependencies to verify setup
from multiAgentSystem.deps import get_deps
from multiAgentSystem.config import LLM_ENDPOINT_NAME, MAX_OUTER_ITERATIONS

print(f"LLM Endpoint: {LLM_ENDPOINT_NAME}")
print(f"Max Iterations: {MAX_OUTER_ITERATIONS}")
print("\nThe RCAAgent class is defined in the cell below. Run that cell first, then use AGENT.predict()")

In [0]:
# This cell should be run AFTER the cell that defines RCAAgent and AGENT
# Make sure to run the cell below that contains the RCAAgent class definition first

# Example request
req = {
    "user_context": (
        """
        Some queries in Databricks take very long to run, at times it looks like it is just idle. But, at times it is quite fast. The Query ID is 01f0a416-cb80-1228-9eda-e3118e89fd48. 
        Task:
        Why is the query taking so long? Were there any problems? If yes, figure out what the problem is and explain it to me with log evidence to support your observations
        """
    ),
    "logs_path": "/Volumes/amruthcatalogtest/default/testsparklogs/00761119/Longer-Bad-00761119_spark/"
}

# Uncomment the following lines after running the RCAAgent definition cell:
# custom_result = AGENT.predict(req)
# custom_result

print("Request prepared. Run the RCAAgent definition cell below first, then uncomment the lines above to execute.")

## Example Usage

Here's an example of how to use the RCA Agent:


In [ ]:
"""Utilities for constructing and running the RCA agent."""
from __future__ import annotations

from typing import Dict, Any, Generator

from multiAgentSystem.state import AgentState

from multiAgentSystem.graph import build_graph
from multiAgentSystem.deps import get_deps
from multiAgentSystem.config import (
    MLFLOW_ENABLED,
    MAX_OUTER_ITERATIONS,
    MAX_ANALYZE_PARSE_LOOPS,
    CONFIDENCE_THRESHOLD,
)


class RCAAgent:
    """Thin wrapper exposing ``predict`` and ``predict_stream`` helpers."""

    def __init__(self, graph=None):
        self.graph = graph or build_graph()

    def _init_state(self, request: Dict[str, Any]) -> AgentState:
        """Initialise the agent state from the inbound request payload."""
        user_context = request.get("user_context")
        if not user_context:
            msgs = request.get("input") or []
            if isinstance(msgs, list):
                user_parts = [
                    m.get("content", "")
                    for m in msgs
                    if isinstance(m, dict) and m.get("role") == "user"
                ]
                user_context = "\n\n".join([p for p in user_parts if p]) or ""

        logs_path = request.get("logs_path", "") or request.get("path", "") or ""

        return AgentState(
            user_context=user_context or "",
            logs_path=logs_path,
            iteration=0,
            hypotheses=[],
            keywords=[],
            evidence=[],
            last_logs_chunk="",
            analyzer_satisfied=False,
            last_generated_keywords=[],  # Initialize the missing field
            draft={"problem": "", "rca": "", "mitigation": ""},
            confidence=0.0,
            critic_approved=False,
            critique="",
            last_status="",
            next_action="",
            supervisor_rationale="",
            analyze_parse_loops=0,
        )

    def predict(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """Run the multi-agent system and return final results."""
        final_state: AgentState = self.graph.invoke(self._init_state(request))
        return {
            "output": {
                "problem": final_state.get("draft", {}).get("problem", ""),
                "rca": final_state.get("draft", {}).get("rca", ""),
                "mitigation": final_state.get("draft", {}).get("mitigation", ""),
                "confidence": float(final_state.get("confidence", 0.0)),
                "iterations": int(final_state.get("iteration", 0)),
                "keywords": final_state.get("keywords", []),
                "evidence": final_state.get("evidence", []),
                "critic_approved": bool(final_state.get("critic_approved", False)),
                "critique": final_state.get("critique", ""),
                "supervisor_rationale": final_state.get("supervisor_rationale", ""),
            }
        }

    def predict_stream(self, request: Dict[str, Any]) -> Generator[Dict[str, Any], None, None]:
        """Run the multi-agent system and yield progress events."""
        state = self._init_state(request)
        for ev in self.graph.stream(state, stream_mode="updates"):
            event_type = ev.get("event")
            node = ev.get("name")
            data = ev.get("data", {}) or {}
            yield {"type": event_type, "node": node, "data": data}

        yield {"type": "final", "node": None, "data": self.predict(request)}


def _maybe_enable_mlflow(agent: RCAAgent) -> None:
    """Enable MLflow autologging for the provided agent if dependencies exist."""
    # Attempt to enable MLflow autologging unconditionally (Databricks environments
    # generally have MLflow available). Failure is best-effort and will not raise.
    try:
        deps = get_deps()
        # If mlflow is installed and available via dependencies, configure autolog.
        deps.mlflow.langchain.autolog()
        deps.mlflow.models.set_model(agent)
    except Exception:
        # Best-effort: failure to enable MLflow should not block agent use.
        pass


# Public convenience exports -------------------------------------------------
AGENT = RCAAgent()
_maybe_enable_mlflow(AGENT)

sample_request = {
    "user_context": (
        "After increasing executor memory and enabling AQE, the nightly ETL job intermittently fails. "
        "Symptoms include long GC pauses and 'executor lost' messages around the shuffle stage."
    ),
    "logs_path": "s3://company-bucket/prod/spark-logs/job-1234/",
}

__all__ = [
    "RCAAgent",
    "AGENT",
    "sample_request",
    "MLFLOW_ENABLED",
    "MAX_OUTER_ITERATIONS",
    "MAX_ANALYZE_PARSE_LOOPS",
    "CONFIDENCE_THRESHOLD",
]
